In [ ]:
# Bootstrap: i notebook vivono in finetuning/ ma tutto il progetto (import,
# data/, results/, main.py) e' relativo alla root del repo -> ci spostiamo li'.
import os, sys
from pathlib import Path

if not Path("main.py").exists():
    os.chdir(Path.cwd().parent)
assert Path("main.py").exists(), "Esegui il notebook dalla root del repo o da finetuning/"
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
print(f"Working directory: {Path.cwd()}")

# 1 - Zero-shot test on LoveDA

Since the generic EoMT-S is trained on **COCO panoptic**, we test it on **LoveDA** without any **additional training**; the results will act as a baseline for the fine-tuning in the later docuemnts.  

The only issue is that we have different classes between the datasets, so we have decided to map the 133 classes of **COCO** into the 7 classes of **LoveDA**, i.e. `house`, `building-other-merged`, `roof` → `building`.  

We expect a value of mIoU relativly low, as in addition of the **arbitrary** mapping of the classes, there is also the difference in images (from the top vs frontal)

In [ ]:
from pathlib import Path

DATA_DIR = Path("data/loveda")
CKPT_PATH = Path("checkpoints/coco_panoptic_eomt_small_640_2x.bin")
RESULTS_PATH = Path("results/baseline_generico.json")

assert DATA_DIR.exists() and CKPT_PATH.exists(), "Need 0_Setup.ipynb"

## 1. COCO → LoveDA

Defined in `finetuning/coco_classes.py`

In [ ]:
import pandas as pd

from finetuning.coco_classes import LOVEDA_FROM_COCO_NAMES, build_coco_to_loveda_mapping
from finetuning.loveda import CLASS_NAMES

display(pd.DataFrame(
    [(k, ", ".join(v)) for k, v in LOVEDA_FROM_COCO_NAMES.items()],
    columns=["LoveDA classes", "COCO class mapping"],
))

coco_to_loveda = build_coco_to_loveda_mapping(CLASS_NAMES)
print(f"COCO classes that became background: {(coco_to_loveda == 0).sum().item()} / 133")

## 2. Building the model & loading checkpoint

Model has **134 outputs** (133 classes + no-object) for matching the checkpoint; used `masked_attn_enabled=False` as done in the paper.

In [ ]:
from finetuning.evaluator import build_eomt_small, load_network_weights

network = build_eomt_small(num_classes=133)
network = load_network_weights(network, CKPT_PATH)

## 3. Evaluation on validation set of LoveDA

In [ ]:
from datasets.loveda_semantic import LoveDASemantic
from finetuning.evaluator import evaluate_on_loveda, save_results

dm = LoveDASemantic(path=str(DATA_DIR), batch_size=2, num_workers=2).setup()

results = evaluate_on_loveda(
    network,
    dm,
    class_mapping=coco_to_loveda,
    max_batches=None,
)
save_results(results, RESULTS_PATH)

In [ ]:
import pandas as pd

df = pd.DataFrame(
    {"IoU (%)": {k: v * 100 for k, v in results["iou_per_class"].items()}}
).round(2)
df.loc["— mIoU —"] = results["miou"] * 100
display(df)

## 4. Samples

Saving indices for future comparison.

In [ ]:
import json

from finetuning.evaluator import predict
from finetuning.visualize import get_val_samples, show_samples

SAMPLE_INDICES = [0, 100, 150, 200, 250, len(dm.val_dataset) - 1]
Path("results").mkdir(exist_ok=True)
Path("results/sample_indices.json").write_text(json.dumps(SAMPLE_INDICES))

imgs, targets = get_val_samples(dm, SAMPLE_INDICES)
preds = predict(network.cuda(), imgs, class_mapping=coco_to_loveda)
show_samples(
    imgs, targets, {"(COCO, zero-shot)": preds},
    save_path="results/qualitative_generico.png",
)